Las Vegas reports its casino results to the state every month, down to the game. I wanted to know what actually drives that money, and whether the recovery since 2020 is as strong as the headlines suggest.

> **The question.** What drives Las Vegas Strip casino revenue, and how predictable is it?

| Source | What it has | Coverage |
|---|---|---|
| **Nevada Gaming Control Board** | casino win by area and game | monthly, Jan 2010 – Sep 2026 |
| **LVCVA** (Las Vegas Convention and Visitors Authority) | visitors, conventions, occupancy, room rates | monthly from 2019; annual from 1970 |
| **Bureau of Labor Statistics** | consumer price index (CPI-U) | monthly |
| **Census Bureau** | Clark County population | annual |

"Win" is what casinos keep after paying out winning bets. It's gaming revenue before expenses. The board publishes its reports as PDFs, so the first job was a parser. It turned 198 monthly reports into a table and checked each month against the report's own totals. Statewide, Strip and downtown figures pass in every month the parser could read; April and May 2020 are scanned images, so I filled those from the LVCVA. The data model and every metric below are SQL (DuckDB), queried directly from this page. **Money is in 2025 dollars** wherever I compare years, because otherwise a decade of inflation looks like growth.

In [1]:
#| collapse
import duckdb
import numpy as np
import pandas as pd
import altair as alt

con = duckdb.connect("../data/vegas.duckdb", read_only=True)
q = lambda sql: con.sql(sql).df()          # run SQL, get a DataFrame

COVID = pd.DataFrame({"start": [pd.Timestamp("2020-03-01")], "end": [pd.Timestamp("2021-06-01")]})
BLUE, ORANGE, AQUA, YELLOW, MUTED = "#3987e5", "#d95926", "#199e70", "#c98500", "#747a87"
q("SELECT min(month) AS first_month, max(month) AS latest_month FROM v_area_monthly WHERE area_key = 'strip'")

,first_month,latest_month
0,2010-01-01,2026-09-01


## Sixteen years on the Strip, in today's dollars

In each year's own dollars, Strip revenue went from about \$6.6 billion in 2019 to \$8.8 billion in 2025, up a third. But prices rose about 26% over the same years, so most of that increase is inflation.

In [2]:
annual = q('''
    SELECT year, round(win / 1e9, 2) AS "Win $B",
           round(real_win / 1e9, 2) AS "In 2025 $"
    FROM v_area_annual
    WHERE area_key = 'strip' AND months_reported = 12 AND year IN (2010, 2013, 2019, 2023, 2024, 2025)
    ORDER BY year
''').set_index("year")
annual

,Win $B,In 2025 $
year,,
2010,5.78,8.53
2013,6.51,9.00
2019,6.59,8.30
2023,8.90,9.41
2024,8.81,9.05
2025,8.82,8.82


In [3]:
#| hide
#| caption: Monthly Strip win in that month's dollars (bars), with the 12-month average in that year's dollars (grey) and in 2025 dollars (blue). The shaded band is the COVID period.
strip = q('''
    SELECT month, total_win / 1e6 AS win_m, win_12m / 12e6 AS avg_nominal, real_win_12m / 12e6 AS avg_real
    FROM v_area_monthly WHERE area_key = 'strip' ORDER BY month
''')
shade = alt.Chart(COVID).mark_rect(color=MUTED, opacity=0.12).encode(x="start:T", x2="end:T")
bars = alt.Chart(strip).mark_bar(color=BLUE, opacity=0.35, width=2).encode(
    x=alt.X("month:T", title=None), y=alt.Y("win_m:Q", title="Strip win per month ($M)"),
    tooltip=[alt.Tooltip("month:T", format="%b %Y", title="Month"), alt.Tooltip("win_m:Q", format=",.0f", title="Win ($M)")])
lines = alt.Chart(strip.melt("month", ["avg_nominal", "avg_real"], "basis", "value").dropna()).mark_line(strokeWidth=2.2).encode(
    x="month:T", y="value:Q",
    color=alt.Color("basis:N", title=None, scale=alt.Scale(domain=["avg_nominal", "avg_real"], range=[MUTED, BLUE]),
                    legend=alt.Legend(labelExpr="datum.value == 'avg_real' ? '12-month average, 2025 dollars' : '12-month average, that year’s dollars'")),
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), alt.Tooltip("value:Q", format=",.0f", title="Average ($M)")])
(shade + bars + lines).properties(title="Adjusted for inflation, the Strip's recovery is a modest step up", height=300)

alt.LayerChart(...)

In 2025 dollars, the Strip earned about **\$8.3 billion in 2019 and \$8.8 billion in 2025, about 6% more**. Real revenue peaked in 2023 at about 13% above 2019 and has eased since. Statewide is similar: about \$15.2 billion in 2019 against \$16.0 billion in 2024, roughly 6% more. (2024 is the latest complete statewide year.)

Inflation also hides something about the 2010s. **Real Strip revenue fell every year** from about \$9.0 billion in 2013 to \$8.3 billion in 2019, while the dollar figures crept up. So the Strip has settled at a higher level since COVID, but only modestly higher than a decade ago, and 2025 is still below 2013.

## Fewer visitors than 2019, more gaming per visit

The LVCVA's annual history goes back further than the monthly data, so I can compare visitors across the whole period.

In [4]:
visitors = q('''
    SELECT year, round(visitors / 1e6, 1) AS "Visitors (M)",
           round(real_win_per_visitor) AS "Strip win per visitor (2025 $)"
    FROM v_visitors_annual ORDER BY year
''').set_index("year")
visitors.T

year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Visitors (M),37.3,38.9,39.7,39.7,41.1,42.3,42.9,42.2,42.1,42.5,19.0,32.2,38.8,40.8,41.7,38.5
Strip win per visitor (2025 $),229.0,223.0,219.0,227.0,211.0,204.0,199.0,201.0,201.0,195.0,244.0,260.0,235.0,230.0,217.0,229.0


In [5]:
#| hide
#| caption: Visitors to Las Vegas per year (LVCVA). Grey bars are the 2020 closure and 2021 reopening.
v = visitors.reset_index().rename(columns={"Visitors (M)": "visitors_m", "Strip win per visitor (2025 $)": "per_visitor"})
v["period"] = np.where(v.year.isin([2020, 2021]), "COVID", "normal")
alt.Chart(v).mark_bar(size=14).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0, values=list(range(2010, 2026, 3)))),
    y=alt.Y("visitors_m:Q", title="Visitors (millions)"),
    color=alt.Color("period:N", legend=None, scale=alt.Scale(domain=["normal", "COVID"], range=[ORANGE, "#b9b6ad"])),
    tooltip=["year:O", alt.Tooltip("visitors_m:Q", title="Visitors (M)", format=".1f")]
).properties(height=200, title="Visitors recovered, then dipped in 2025")

alt.Chart(...)

In [6]:
#| hide
#| caption: Strip win divided by all Las Vegas visitors, in 2025 dollars. The 2020–21 spike is the closure: visitor counts fell further than gaming, probably because locals and drive-in players kept playing.
alt.Chart(v).mark_line(color=BLUE, strokeWidth=2, point=alt.OverlayMarkDef(size=60, filled=True, color=BLUE)).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0, values=list(range(2010, 2026, 3)))),
    y=alt.Y("per_visitor:Q", title="2025 dollars", scale=alt.Scale(domain=[180, 270])),
    tooltip=["year:O", alt.Tooltip("per_visitor:Q", title="Win per visitor (2025 $)", format=",.0f")]
).properties(height=220, title="Strip win per visitor fell through the 2010s, then recovered")

alt.Chart(...)

Visitors in 2022–2024 (39–42 million a year) were close to mid-2010s levels. The clearer drop is **2025, down about 8% from 2024**, back to roughly where 2011 was.

Real win per visitor is the more interesting number. It **fell through the 2010s**, from about \$229 in 2010 to \$195 in 2019, as visitor numbers grew but revenue didn't keep pace. Since reopening, it's been about \$217–235, close to early-2010s levels. So the Strip isn't getting record amounts out of each visit. It's recovered what it had been losing.

One caution: visitor counts cover all of Las Vegas, not just the Strip. So this is an index of how much gaming each visit supports, not what any one person spends.

## Rooms cost more, and fewer are filled

Monthly room data starts in 2019, so this covers the recovery but not the decade before. 2026 is a partial year (January–September).

In [7]:
rooms = q('''
    SELECT year, round(adr) AS "Room rate $", round(real_adr) AS "In 2025 $",
           round(occupancy * 100, 1) AS "Occupancy %"
    FROM v_strip_rooms_annual ORDER BY year
''').set_index("year")
rooms

,Room rate $,In 2025 $,Occupancy %
year,,,
2019,143.0,181.0,90.4
2020,113.0,141.0,42.1
2021,139.0,165.0,67.7
2022,181.0,199.0,81.6
2023,204.0,216.0,86.2
2024,206.0,212.0,86.4
2025,197.0,197.0,83.2
2026,194.0,188.0,83.1


In 2025 dollars, the Strip's average nightly rate rose from about **\$181 in 2019 to \$216 in 2023 (+19%)**, while occupancy fell from 90% to 86%. Rates have eased since (\$197 in 2025), with occupancy around 83%. Fewer, pricier room nights fits a market leaning on price over volume. But this data can't tell me whether resorts chose that or demand simply softened. And without monthly rates before 2019, I can't say whether rates had been falling through the 2010s the way per-visitor gaming had.

## What do locals contribute?

No public data splits Strip bets between residents and tourists. But Nevada reports win for the **locals market**: casinos in North Las Vegas, the Boulder Strip and the rest of Clark County (Henderson and the suburbs), which mostly serve residents. COVID also created something close to a natural experiment. From mid-2020 into 2021, residents could gamble, but most tourists, especially international ones, stayed away.

In [8]:
rec = q("SELECT month, locals_vs_2019, strip_vs_2019, strip_slots_vs_2019, strip_tables_vs_2019 FROM v_covid_recovery WHERE month >= '2020-06-01'")
rec[rec.month.dt.month.isin([7, 8])].set_index("month").round(2)

,locals_vs_2019,strip_vs_2019,strip_slots_vs_2019,strip_tables_vs_2019
month,,,,
2020-07-01,0.92,0.61,0.62,0.60
2021-07-01,1.13,1.46,1.39,1.56
2020-08-01,1.10,0.61,0.67,0.55
2021-08-01,1.41,1.20,1.38,1.02


In [9]:
#| hide
#| caption: Each month's win as a share of the same month in 2019 (1.0 = back to normal), June 2020 to December 2021.
long = rec.melt("month", ["locals_vs_2019", "strip_slots_vs_2019", "strip_tables_vs_2019"], "series", "ratio")
names = {"locals_vs_2019": "Locals market", "strip_slots_vs_2019": "Strip slots", "strip_tables_vs_2019": "Strip tables"}
long["series"] = long.series.map(names)
one = alt.Chart(pd.DataFrame({"y": [1]})).mark_rule(strokeDash=[4, 4], color=MUTED).encode(y="y:Q")
lines = alt.Chart(long).mark_line(strokeWidth=2.2, point=True).encode(
    x=alt.X("month:T", title=None), y=alt.Y("ratio:Q", title="Share of same month in 2019", axis=alt.Axis(format="%")),
    color=alt.Color("series:N", title=None, scale=alt.Scale(domain=list(names.values()), range=[AQUA, BLUE, ORANGE])),
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), "series:N", alt.Tooltip("ratio:Q", format=".0%")])
(one + lines).properties(title="Locals came back in weeks; the Strip took most of a year", height=280)

alt.LayerChart(...)

The locals market was back to roughly 2019 levels by **July–August 2020**, while the Strip was still around 60%. That's what you'd expect from casinos that run on residents. On the Strip, **slots recovered faster than tables**: about 67–73% of normal in August–September 2020, against roughly half for tables. That fits slot play leaning on domestic and drive-in visitors, while baccarat depends on international high rollers who couldn't travel.

Does a growing population explain the Strip's gains? Here's win per resident, in 2025 dollars:

In [10]:
res = q('''
    SELECT year, round(population / 1e6, 2) AS "Residents (M)", round(locals_per_resident) AS "Locals $ / resident",
           round(strip_per_resident) AS "Strip $ / resident"
    FROM v_market_per_resident WHERE year IN (2010, 2015, 2019, 2022, 2024) ORDER BY year
''').set_index("year")
res

,Residents (M),Locals $ / resident,Strip $ / resident
year,,,
2010,1.95,1544.0,4371.0
2015,2.10,1390.0,4114.0
2019,2.28,1348.0,3647.0
2022,2.32,1391.0,3923.0
2024,2.39,1357.0,3793.0


Clark County grew about 22% from 2010 to 2024, to 2.4 million people. Real locals-market win grew only about 7%, so **locals-market win per resident fell**, from about \$1,540 to \$1,360 a year. Strip win per resident fell too. Population grew faster than the Strip's revenue, so it can't explain the Strip's gains. This doesn't show how much residents spend on the Strip itself, but the Strip's fortunes look like they rest on visitors.

## Slots pull ahead

The Strip used to be close to an even split between table games and slot machines. Now slots earn more.

In [11]:
# Shares of each year's win, using only months with game-level detail
# (three 2025 months have totals only, so raw sums would understate that year).
mix = q('''
    WITH grouped AS (
        SELECT year(month) AS year,
               CASE WHEN game_group IN ('Roulette', 'Poker-style table games', 'Sports & race', 'Poker room', 'Other tables')
                    THEN 'Other table games' ELSE game_group END AS game_group,
               sum(win) AS win
        FROM v_game_mix
        WHERE area_key = 'strip' AND year(month) BETWEEN 2010 AND 2025
        GROUP BY 1, 2
    )
    SELECT year, game_group, win / sum(win) OVER (PARTITION BY year) AS share FROM grouped
''')
slot_share = mix[mix.game_group.str.contains("slots")].groupby("year").share.sum()
slot_share.loc[[2010, 2015, 2019, 2020, 2021, 2025]].round(3).to_frame("Slot share of Strip win")

,Slot share of Strip win
year,
2010,0.483
2015,0.482
2019,0.523
2020,0.537
2021,0.564
2025,0.562


In [12]:
#| hide
#| caption: Each game group's share of annual Strip win. Multi-denomination slots are the modern video machines that let a player choose the bet size.
order = ["Multi-denomination slots", "Penny slots", "High-limit slots ($5+)", "Other slots",
         "Baccarat", "Blackjack", "Craps", "Other table games"]
colors = ["#2340ff", "#6a7cff", "#a3aeff", "#d3d8ff", "#eb6834", "#eda100", "#1baf7a", "#b9b6ad"]
alt.Chart(mix).mark_area().encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0, values=[2010, 2013, 2016, 2019, 2022, 2025])),
    y=alt.Y("share:Q", title="Share of Strip win", stack="normalize", axis=alt.Axis(format="%")),
    color=alt.Color("game_group:N", title=None, sort=order, scale=alt.Scale(domain=order, range=colors),
                    legend=alt.Legend(columns=2, symbolType="square")),
    order=alt.Order("order_idx:Q"),
    tooltip=["year:O", alt.Tooltip("game_group:N", title="Group"), alt.Tooltip("share:Q", format=".1%", title="Share")],
).transform_calculate(order_idx="indexof(" + str(order) + ", datum.game_group)").properties(
    title="Slots (blues) now earn more than half the Strip's win", height=320)

alt.Chart(...)

Slots went from about **48% of Strip win in 2010 to 56% in 2025**. About half of that shift happened before COVID, between 2015 and 2019. The rest came in 2020–2021, when international baccarat players couldn't travel and slot players could. Once travel came back, slots kept the larger share.

The growth is in multi-denomination video slots: about \$1.3 billion in 2019 to \$3.4 billion in 2024, in each year's dollars. Penny slots nearly doubled through 2019 and then fell sharply. Part of that swap is probably classification rather than behavior. Modern machines offer several bet sizes, so play that used to be recorded as "penny" now lands in "multi-denomination."

## Fewer blackjack tables, more baccarat tables

The board reports how many tables of each game operate, so I can look at supply as well as revenue.

In [13]:
tables = q('''
    SELECT game, year, round(tables) AS tables, round(real_win_per_table_month / 1000) AS win_per_table_k
    FROM v_table_economics WHERE game IN ('Blackjack', 'Baccarat', 'Craps', 'Roulette') AND year IN (2010, 2019, 2025)
''')
tables.pivot(index="game", columns="year", values=["tables", "win_per_table_k"])

tables                 win_per_table_k              
year         2010    2019    2025            2010   2019   2025
game                                                           
Baccarat    242.0   333.0   393.0           603.0  327.0  298.0
Blackjack  1405.0  1222.0  1036.0            63.0   71.0   73.0
Craps       202.0   182.0   180.0           163.0  175.0  152.0
Roulette    274.0   266.0   267.0           116.0  122.0  108.0

*Win per table is per month, in thousands of 2025 dollars.*

- **Blackjack:** the Strip runs about **26% fewer tables** than in 2010 (roughly 1,400 to 1,040), and real win per table is about 16% higher. Fewer tables each earning more is consistent with higher table minimums concentrating play. Industry reporting says minimums have risen, but the board's data doesn't include them.
- **Baccarat:** **62% more tables** (about 240 to 390), but real win per table has roughly halved. Real baccarat win is actually lower than in 2010, so casinos added capacity while the play itself shrank.
- **Craps and roulette:** table counts and real win per table have moved by roughly a tenth or less since 2010.

So the drop in blackjack tables is a supply decision, but this data can't say which way it runs. Casinos may have cut tables because players left, or players may have left because of higher minimums. Answering that would take minimum-bet surveys or casino-level data.

## Baccarat swings the monthly numbers

Baccarat is played for very high stakes by a small number of players. The house's long-run edge is small, so in any one month the result depends heavily on whether a few big players win or lose. The share of chips bought that the house ends up keeping is called the **hold**.

In [14]:
# How much does each line move from one month to the next? (median absolute change, 2022 on)
swing = q('''
    WITH g AS (
        SELECT game_group, month, win / LAG(win) OVER (PARTITION BY game_group ORDER BY month) - 1 AS change
        FROM v_game_mix
        WHERE area_key = 'strip' AND month >= '2022-01-01'
          AND game_group IN ('Baccarat', 'Blackjack', 'Multi-denomination slots')
    )
    SELECT game_group AS "Game group",
           median(abs(change))              AS "Typical monthly change",
           quantile_cont(abs(change), 0.9)  AS "1 month in 10 moves at least"
    FROM g GROUP BY ALL ORDER BY 2 DESC
''').set_index("Game group")
swing.round(2)

,Typical monthly change,1 month in 10 moves at least
Game group,,
Baccarat,0.29,0.97
Blackjack,0.17,0.44
Multi-denomination slots,0.07,0.16


In [ ]:
#| label: Share of month-to-month variation in Strip win from baccarat, 2022 on
# How much of the Strip's month-to-month change comes from baccarat? (2022 on)
mv = q('''
    SELECT a.month, a.total_win, g.win AS baccarat
    FROM v_area_monthly a
    LEFT JOIN v_game_mix g ON g.month = a.month AND g.area_key = 'strip' AND g.game_group = 'Baccarat'
    WHERE a.area_key = 'strip' AND a.month >= '2022-01-01'
    ORDER BY a.month
''')
d = mv[["total_win", "baccarat"]].diff().dropna()   # months without game detail drop out
round(d.cov().loc["baccarat", "total_win"] / d["total_win"].var(), 2)

np.float64(0.52)

In [15]:
#| hide
#| caption: Monthly baccarat hold on the Strip (dots) against its trailing two-year average (line), 2010 to 2026. Hover for the month's baccarat win.
bac = q("SELECT month, hold, hold_trailing_24m, baccarat_win / 1e6 AS win_m FROM v_baccarat ORDER BY month")
pts = alt.Chart(bac).mark_circle(size=36, color=ORANGE, opacity=0.85).encode(
    x=alt.X("month:T", title=None),
    y=alt.Y("hold:Q", title="Hold (share of chips kept)", axis=alt.Axis(format="%")),
    tooltip=[alt.Tooltip("month:T", format="%b %Y", title="Month"), alt.Tooltip("hold:Q", format=".1%", title="Hold"),
             alt.Tooltip("win_m:Q", format=",.0f", title="Baccarat win ($M)")])
line = alt.Chart(bac.dropna(subset=["hold_trailing_24m"])).mark_line(color="#e8e9ec", strokeWidth=2).encode(x="month:T", y="hold_trailing_24m:Q")
(line + pts).properties(title="Baccarat hold has always swung, and the average has crept up", height=280)

alt.LayerChart(...)

In [16]:
by_year = q('''
    SELECT year, round(typical_change, 2) AS "Typical monthly change in win",
           round(hold_min * 100, 1) AS "Lowest hold (%)", round(hold_max * 100, 1) AS "Highest hold (%)"
    FROM v_baccarat_by_year WHERE n_months = 12 ORDER BY year
''').set_index("year")
by_year.T

year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024
Typical monthly change in win,0.65,0.43,0.43,0.44,0.26,0.29,0.33,0.3,0.27,0.27,0.47,0.22,0.35,0.36
Lowest hold (%),3.50,9.70,8.00,7.60,9.00,8.30,10.20,10.0,8.80,7.70,6.50,8.90,7.10,8.80
Highest hold (%),18.60,19.00,16.10,18.90,16.60,17.50,18.10,17.1,16.50,20.90,22.20,22.10,23.90,20.80


Since 2022, baccarat win typically moves about **29%** from one month to the next, against **7%** for multi-denomination slots, so it's roughly four times as jumpy. One month in ten, it moves by almost 100%. Baccarat is about a sixth of Strip revenue, but it accounts for **about half of the month-to-month variation** in the total.

**The volatility changes over the years, though.** Month-to-month swings were largest in **2010–2013** (typically 43–65%). They calmed to about 30% from 2014 to 2019. Since 2019, hold itself has spread wider, with single months ranging from under 3% to almost 27%. The two-year average hold has also risen, from about 11–12% in the early 2010s to around 16%.

So when a monthly headline says "Strip revenue fell," the first question to ask is **what did baccarat do?**

## Seasonality

Each month's revenue compared with its year's monthly average, across 14 normal years (2010–2019 and 2022–2025).

In [17]:
season = q("SELECT month_num, month_name, index, index_min, index_max FROM v_seasonality WHERE area_key = 'strip' ORDER BY month_num")
season.set_index("month_name")[["index"]].T.round(2)

month_name,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
index,1.01,1.05,0.98,0.88,1.01,0.96,1.03,0.97,1.0,1.02,0.98,1.12


In [18]:
#| hide
#| caption: Average seasonal index (dots) with the range across 14 years (lines). 1.0 is an average month.
base = alt.Chart(season).encode(x=alt.X("month_name:N", sort=list(season.month_name), title=None, axis=alt.Axis(labelAngle=0)))
rng = base.mark_rule(color=MUTED, strokeWidth=2).encode(y=alt.Y("index_min:Q", title="Index (1 = average month)", scale=alt.Scale(domain=[0.75, 1.3])), y2="index_max:Q")
dot = base.mark_point(size=90, filled=True, color=BLUE).encode(
    y="index:Q", tooltip=[alt.Tooltip("month_name:N", title="Month"), alt.Tooltip("index:Q", format=".2f", title="Average"),
                          alt.Tooltip("index_min:Q", format=".2f", title="Lowest"), alt.Tooltip("index_max:Q", format=".2f", title="Highest")])
one = alt.Chart(pd.DataFrame({"y": [1]})).mark_rule(strokeDash=[4, 4], color=MUTED).encode(y="y:Q")
(one + rng + dot).properties(title="December peaks, April dips", height=260)

alt.LayerChart(...)

**December is the strongest month**, about 12% above average, and **April is the weakest**, about 12% below. The ranges are wide, though. Baccarat probably accounts for much of that, since Lunar New Year can land anywhere from late January to mid-February. So seasonality is a tendency, not a rule.

## How predictable is it?

I trained each model on January 2022 to September 2025, after the reopening rebound, and scored it on the following twelve months, which it never saw. The score is the average miss as a percentage of the actual figure (MAPE).

- **Seasonal naive:** next October = last October. It's the baseline any model has to beat.
- **Naive with growth:** the same, scaled by the last year's growth.
- **Holt-Winters:** exponential smoothing with a damped trend and a seasonal pattern.

In [19]:
#| collapse
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings("ignore")

s = q("SELECT month, total_win, slots_win FROM v_area_monthly WHERE area_key = 'strip' AND month >= '2022-01-01' ORDER BY month")
s = s.set_index(pd.DatetimeIndex(s.month, freq="MS"))
s["slots_win"] = s["slots_win"].interpolate()   # 3 months in 2025 have totals only

def score(y):
    train, test = y[:"2025-09"], y["2025-10":]
    mape = lambda p: float(np.mean(np.abs(p.values - test.values) / test.values))
    naive = y.shift(12)["2025-10":]
    growth = train[-12:].sum() / train[-24:-12].sum()
    hw = ExponentialSmoothing(train, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12).fit()
    return {"Seasonal naive": mape(naive), "Naive with growth": mape(naive * growth),
            "Holt-Winters": mape(hw.forecast(len(test)))}

scores = pd.DataFrame({"Total win": score(s.total_win), "Slots only": score(s.slots_win)})
scores.map(lambda v: f"{v:.1%}")

,Total win,Slots only
Seasonal naive,5.4%,5.0%
Naive with growth,5.7%,4.7%
Holt-Winters,3.8%,3.1%


In [20]:
#| hide
#| caption: Holt-Winters refit on all data since 2022, forecasting six months ahead. The band is ± twice the typical holdout error, a rough guide rather than a formal interval.
hw_all = ExponentialSmoothing(s.total_win, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12).fit()
fc = hw_all.forecast(6)
err = float(score(s.total_win)["Holt-Winters"])
fdf = pd.DataFrame({"month": fc.index, "win_m": fc.values / 1e6})
fdf["lo"], fdf["hi"] = fdf.win_m * (1 - 2 * err), fdf.win_m * (1 + 2 * err)
hist = pd.DataFrame({"month": s.index, "win_m": s.total_win.values / 1e6})
hist = hist[hist.month >= "2024-01-01"]
band = alt.Chart(fdf).mark_area(color=BLUE, opacity=0.18).encode(x="month:T", y="lo:Q", y2="hi:Q")
actual = alt.Chart(hist).mark_line(color="#e8e9ec", strokeWidth=2).encode(
    x=alt.X("month:T", title=None), y=alt.Y("win_m:Q", title="Strip win ($M)", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), alt.Tooltip("win_m:Q", format=",.0f", title="Actual ($M)")])
pred = alt.Chart(fdf).mark_line(color=BLUE, strokeWidth=2, strokeDash=[5, 3]).encode(
    x="month:T", y="win_m:Q",
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), alt.Tooltip("win_m:Q", format=",.0f", title="Forecast ($M)"),
             alt.Tooltip("lo:Q", format=",.0f", title="Low"), alt.Tooltip("hi:Q", format=",.0f", title="High")])
(band + actual + pred).properties(title="Next six months: forecast (dashed) after actuals", height=280)

alt.LayerChart(...)

Holt-Winters beats both naive baselines, missing by under 4% a month on total win and about 3% on slots alone. That's useful for planning, but it's only about 1.5 points better than "same as last year." The Strip is a mature, seasonal business, so last year is already a good guess. Some of the remaining error is baccarat, which no model can call month to month, but slots alone still miss by about 3%.

These scores come from a single twelve-month test, so they're a rough read, not a benchmark. One test month (October 2025) has totals only, so its slot figure is interpolated. Forecasting slots and tables separately, and treating baccarat as a range rather than a number, would be the honest next step.

## What it adds up to

1. **The recovery is real but modest once you remove inflation:** about 6% above 2019 in 2025 dollars, after real revenue fell every year from 2013 to 2019.
2. **Visitors came back near mid-2010s levels**, then dipped in 2025. **Gaming per visitor recovered to early-2010s levels** after falling through the 2010s.
3. **Rooms cost more and fewer are filled:** real room rates rose about 19% from 2019 to 2023, then eased.
4. **Locals carry the locals casinos, but they don't explain the Strip's gains.** The locals market came back within weeks of reopening, but real win per resident has fallen as the valley grew.
5. **Slots are now the base of the business.** Their share was already rising before 2020, and it jumped when international baccarat travel stopped.
6. **Read monthly headlines through baccarat.** It's about a sixth of revenue but half of the month-to-month variation, and a single month is often luck at a few high-stakes tables.

## What this does not show

- Area totals hide individual properties, so I can't separate one resort from another.
- "Win" is revenue before expenses. Free play, comps and promotions sit outside it, so margins aren't visible.
- Visitor counts cover all of Las Vegas, so win per visitor is an index, not personal spending. No public data splits Strip bets between residents and tourists.
- Inflation uses the national CPI-U. Las Vegas prices may have moved differently.
- Table minimums and buy-ins aren't in the board's reports.
- April and May 2020 come from the LVCVA because the board's reports are scanned images. Three 2025 months have totals but no game detail.
- The forecast scores come from one twelve-month test window.

The pipeline, SQL model and data tests are in the [project folder on GitHub](https://github.com/mckeeander/mckeeander.github.io/tree/main/projects/vegas-gaming).

*Sources: Nevada Gaming Control Board, Monthly Gaming Revenue Reports; Las Vegas Convention and Visitors Authority, monthly visitor statistics and Historic Tourism Statistics 1970–2025; U.S. Bureau of Labor Statistics, CPI-U; U.S. Census Bureau, county population estimates.*